In [2]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.metrics import precision_recall_curve, roc_curve, auc, average_precision_score
from pathlib import Path
import json

# Setup paths based on project charter
BASE_DIR = Path("C:/Projects/Majorproject")
PREDS_FILE = BASE_DIR / "results/models/xgboost_f4_v1/validation_predictions.parquet"
MODEL_FILE = BASE_DIR / "research/models/baseline/xgboost_f4_v1/model.json"
FEATURE_CONTRACT = BASE_DIR / "research/models/baseline/xgboost_f4_v1/feature_definition.json"
VAL_SPLIT = BASE_DIR / "data/processed/hi_small/modeling/splits/validation.parquet"
PLOT_DIR = BASE_DIR / "results/plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Load validation predictions
df_preds = pl.read_parquet(PREDS_FILE)
y_true = df_preds["true_label"].to_numpy()
y_scores = df_preds["xgboost_probability"].to_numpy()

c:\Projects\Majorproject\amlPlayground\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)
prevalence = y_true.sum() / len(y_true)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'XGBoost (PR-AUC = {pr_auc:.4f})', color='blue', lw=2)
plt.axhline(y=prevalence, color='red', linestyle='--', label=f'No Skill ({prevalence:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Validation)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.savefig(PLOT_DIR / "pr_curves.png", dpi=300, bbox_inches='tight')
plt.close()

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'XGBoost (ROC-AUC = {roc_auc:.4f})', color='blue', lw=2)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Validation)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.savefig(PLOT_DIR / "roc_curves.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved PR and ROC curves.")

Saved PR and ROC curves.


In [4]:
# Calculate metrics across thresholds
thresholds = np.linspace(0, 1, 100)
precisions, recalls, f1s, alerts = [], [], [], []

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    tp = (preds & y_true).sum()
    fp = (preds & ~y_true).sum()
    fn = (~preds & y_true).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)
    alerts.append(preds.sum())

# Plot 1: Metrics vs Threshold
plt.figure(figsize=(8, 6))
plt.plot(thresholds, precisions, label='Precision', color='blue')
plt.plot(thresholds, recalls, label='Recall', color='green')
plt.plot(thresholds, f1s, label='F1 Score', color='purple')
plt.axvline(x=0.95896, color='black', linestyle='--', label='Best F1 (0.95896)')
plt.axvline(x=0.03219, color='red', linestyle='--', label='80% Recall (0.03219)')
plt.xlabel('Decision Threshold')
plt.ylabel('Metric Score')
plt.title('Precision, Recall, and F1 vs. Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(PLOT_DIR / "threshold_tradeoff.png", dpi=300, bbox_inches='tight')
plt.close()

# Plot 2: Recall vs Alert Volume
plt.figure(figsize=(8, 6))
plt.plot(alerts, recalls, color='blue', lw=2)
plt.xscale('log')
plt.axvline(x=100, color='gray', linestyle='--', label='Top 100 Alerts')
plt.axvline(x=19508, color='red', linestyle='--', label='80% Recall (19,508 Alerts)')
plt.axvline(x=76034, color='orange', linestyle='--', label='95% Recall (76,034 Alerts)')
plt.xlabel('Number of Alerts (Log Scale)')
plt.ylabel('Cumulative Recall')
plt.title('Recall vs. Alert Volume')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(PLOT_DIR / "alerts_vs_recall.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved Threshold and Alert Volume plots.")

Saved Threshold and Alert Volume plots.


In [7]:
# Load Model and Feature Contract
booster = xgb.Booster()
booster.load_model(MODEL_FILE)
with open(FEATURE_CONTRACT, 'r') as f:
    contract = json.load(f)

# Use the correct key:
feature_names = contract["feature_names"]

# Extract Gain
gain_importance = booster.get_score(importance_type='gain')
sorted_gain = sorted(gain_importance.items(), key=lambda x: x[1], reverse=True)[:20]
features_top20 = [x[0] for x in sorted_gain][::-1]
gains_top20 = [x[1] for x in sorted_gain][::-1]

plt.figure(figsize=(10, 8))
plt.barh(features_top20, gains_top20, color='teal')
plt.title('Top 20 XGBoost Features by Gain')
plt.xlabel('Gain')
plt.tight_layout()
plt.savefig(PLOT_DIR / "xgb_feature_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved XGBoost Feature Importance plot.")

Saved XGBoost Feature Importance plot.


In [9]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backend.core.schemas import FeatureRow
from backend.services.features.feature_contract import FeatureContract

TARGET_TX_ID = 3560804 

# 1. Load the feature contract
contract = FeatureContract(FEATURE_CONTRACT)

# 2. Extract the specific row from the validation set using Polars
df_val_pl = pl.read_parquet(VAL_SPLIT).filter(pl.col("transaction_id") == TARGET_TX_ID)

if len(df_val_pl) == 0:
    print(f"Transaction {TARGET_TX_ID} not found in validation split.")
else:
    # 3. Convert raw Polars rows into FeatureRows so the contract can map and one-hot encode them
    raw_dicts = df_val_pl.to_dicts()
    feature_rows = [
        FeatureRow(transaction_id=row["transaction_id"], features=row) 
        for row in raw_dicts
    ]
    
    # 4. Generate the matrix and wrap it in a DataFrame so SHAP knows the column names
    matrix = contract.to_matrix(feature_rows)
    X_target = pd.DataFrame(matrix, columns=contract.feature_names)
    
    # 5. Initialize SHAP explainer
    explainer = shap.TreeExplainer(booster)
    shap_values = explainer(X_target)
    
    # Generate Waterfall Plot
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values[0], max_display=10, show=False)
    plt.title(f'Local TreeSHAP Explanation (TX: {TARGET_TX_ID})')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "shap_local.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved Local SHAP Waterfall plot.")

Saved Local SHAP Waterfall plot.


In [10]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from pathlib import Path

# Setup paths
BASE_DIR = Path("C:/Projects/Majorproject")
TRANSACTION_DIR = BASE_DIR / "data/processed/hi_small/transactions"
PLOT_DIR = BASE_DIR / "results/plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Scanning full canonical dataset for daily volumes...")
# Lazily scan all 11 partitions, aggregate by day, then collect
daily_stats = (
    pl.scan_parquet(TRANSACTION_DIR / "*.parquet")
    .select(["timestamp", "is_laundering"])
    .with_columns(pl.col("timestamp").dt.date().alias("date"))
    .group_by("date")
    .agg([
        pl.len().alias("total_tx"),
        pl.col("is_laundering").sum().alias("laundering_tx")
    ])
    .sort("date")
    .collect()
)

dates = daily_stats["date"].to_list()
total_tx = daily_stats["total_tx"].to_list()
laundering_tx = daily_stats["laundering_tx"].to_list()

# Plotting
fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar chart for Total Transactions
ax1.bar(dates, total_tx, color='#4C72B0', alpha=0.7, label='Total Transactions (Legitimate + Laundering)')
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Total Transaction Volume', color='#4C72B0', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#4C72B0')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Line chart for Laundering Labels on secondary Y-axis
ax2 = ax1.twinx()
ax2.plot(dates, laundering_tx, color='#C44E52', marker='o', linewidth=2.5, markersize=6, label='Laundering Labels')
ax2.set_ylabel('Laundering Labels (Count)', color='#C44E52', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#C44E52')

# Mark Dense vs Sparse Period
dense_end = datetime.date(2022, 9, 10)
ax1.axvline(x=dense_end, color='black', linestyle='--', linewidth=1.5, alpha=0.8)

# Annotate regions
ax1.text(datetime.date(2022, 9, 5), max(total_tx)*0.85, 'Dense Period\n(Sep 1 – Sep 10)', 
         ha='center', va='center', fontsize=11, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
ax1.text(datetime.date(2022, 9, 14), max(total_tx)*0.85, 'Sparse Tail\n(Sep 11 – Sep 18)', 
         ha='center', va='center', fontsize=11, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.title('Daily Transaction Volume and Laundering Labels (HI-Small)', fontsize=14, pad=15)
fig.tight_layout()

out_path_1 = PLOT_DIR / "daily_volume_labels.png"
plt.savefig(out_path_1, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved Figure 4.1 to {out_path_1}")

Scanning full canonical dataset for daily volumes...
Saved Figure 4.1 to C:\Projects\Majorproject\results\plots\daily_volume_labels.png


In [11]:
print("Scanning full canonical dataset for amount distributions...")
# Extract just the amounts (converted to numpy arrays for matplotlib)
# This uses ~40MB of RAM for 5M floats, well within limits
amounts_df = (
    pl.scan_parquet(TRANSACTION_DIR / "*.parquet")
    .select(["amount_paid", "amount_paid_log"])
    .collect()
)

raw_amounts = amounts_df["amount_paid"].to_numpy()
log_amounts = amounts_df["amount_paid_log"].to_numpy()

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Subplot 1: Raw amounts (using log-scale on the Y axis because of the heavy skew)
axes[0].hist(raw_amounts, bins=60, color='#55A868', edgecolor='black', alpha=0.8)
axes[0].set_title('Raw Transaction Amounts (Heavy Skew)', fontsize=12, pad=10)
axes[0].set_xlabel('Amount Paid (Raw)', fontweight='bold')
axes[0].set_ylabel('Frequency (Log Scale)', fontweight='bold')
axes[0].set_yscale('log') # Y-axis logged to see the long tail
axes[0].grid(axis='y', linestyle='--', alpha=0.6)

# Subplot 2: Log amounts (Normally distributed representation)
axes[1].hist(log_amounts, bins=60, color='#8172B3', edgecolor='black', alpha=0.8)
axes[1].set_title('Log Transformed Amounts', fontsize=12, pad=10)
axes[1].set_xlabel('Amount Paid (Log)', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].grid(axis='y', linestyle='--', alpha=0.6)

fig.suptitle('Distribution of Transaction Amounts across the Dataset', fontsize=14, y=1.02)
fig.tight_layout()

out_path_2 = PLOT_DIR / "amount_distribution.png"
plt.savefig(out_path_2, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved Figure 4.2 to {out_path_2}")

Scanning full canonical dataset for amount distributions...
Saved Figure 4.2 to C:\Projects\Majorproject\results\plots\amount_distribution.png


In [12]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from pathlib import Path

# Setup paths
BASE_DIR = Path("C:/Projects/Majorproject")
PLOT_DIR = BASE_DIR / "results/plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Frozen chronological splits from Phase 2B
splits = [
    {"name": "Train", "start": "2022-09-01 00:00", "end": "2022-09-07 14:51", "rows": "3,553,739 rows", "color": "#4C72B0"},
    {"name": "Validation", "start": "2022-09-07 14:52", "end": "2022-09-09 03:13", "rows": "761,646 rows", "color": "#55A868"},
    {"name": "Test", "start": "2022-09-09 03:14", "end": "2022-09-10 23:59", "rows": "761,852 rows", "color": "#C44E52"},
    {"name": "Tail Stress", "start": "2022-09-11 00:05", "end": "2022-09-18 16:18", "rows": "1,108 rows", "color": "#8172B3"}
]

fig, ax = plt.subplots(figsize=(12, 3))

# Draw each split as a colored span
for s in splits:
    start = datetime.datetime.strptime(s["start"], "%Y-%m-%d %H:%M")
    end = datetime.datetime.strptime(s["end"], "%Y-%m-%d %H:%M")
    
    # Span block
    ax.axvspan(start, end, ymin=0.2, ymax=0.8, color=s["color"], alpha=0.8)
    
    # Text labels centered inside the block
    midpoint = start + (end - start) / 2
    ax.text(midpoint, 0.58, s["name"], ha='center', va='center', color='white', fontweight='bold', fontsize=11)
    ax.text(midpoint, 0.42, s["rows"], ha='center', va='center', color='white', fontsize=10)

# Format the X-axis timeline
ax.set_yticks([]) # Hide the y-axis
ax.set_xlim(datetime.datetime(2022, 9, 1), datetime.datetime(2022, 9, 19))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.tick_params(axis='x', labelsize=11)

# Clean up borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.title('Chronological Split Timeline (Sep 1 – Sep 18)', fontsize=14, pad=15)

out_path = PLOT_DIR / "split_timeline.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved Figure 4.3 to {out_path}")

Saved Figure 4.3 to C:\Projects\Majorproject\results\plots\split_timeline.png


In [13]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Setup paths based on project charter
BASE_DIR = Path("C:/Projects/Majorproject")
# LaTeX expects this in the 'figures' directory
FIG_DIR = BASE_DIR / "figures" 
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

# Styling for boxes and arrows
box_style = dict(boxstyle="round,pad=0.6", edgecolor="#4C72B0", facecolor="#EAEAF2", linewidth=2.5)
arrow_props = dict(facecolor='#333333', edgecolor='#333333', arrowstyle="-|>", lw=2.5, mutation_scale=20)

# ==========================================
# Layer 1: Planned Graph-Investigation Layer
# ==========================================
ax.text(0.5, 0.9, "Planned Graph-Investigation Layer", fontsize=15, fontweight='bold', ha='center', color="#333333")

nodes_1 = [
    (0.12, 0.7, "Escalated\nEvent"),
    (0.38, 0.7, "Bounded Temporal\nSubgraph\n(only edges < t)"),
    (0.64, 0.7, "Graph\nAnalytics"),
    (0.88, 0.7, "GraphSAGE\nScorer")
]

for i in range(len(nodes_1)):
    x, y, text = nodes_1[i]
    ax.text(x, y, text, ha='center', va='center', fontsize=12, fontweight='bold', bbox=box_style, color="#333333")
    
    # Draw arrows connecting nodes
    if i < len(nodes_1) - 1:
        next_x, next_y, _ = nodes_1[i+1]
        ax.annotate('', xy=(next_x - 0.1, next_y), xytext=(x + 0.1, y), arrowprops=arrow_props)


# ==========================================
# Layer 2: Model Plug-in Lifecycle
# ==========================================
# Draw a subtle separator line
ax.plot([0.1, 0.9], [0.45, 0.45], color="gray", linestyle="--", alpha=0.5)

ax.text(0.5, 0.35, "Model Plug-in Lifecycle", fontsize=15, fontweight='bold', ha='center', color="#333333")

nodes_2 = [
    (0.12, 0.15, "Colab\nTraining"),
    (0.38, 0.15, "Export\nBundle"),
    (0.64, 0.15, "Local\nValidation"),
    (0.88, 0.15, "Activation")
]

for i in range(len(nodes_2)):
    x, y, text = nodes_2[i]
    ax.text(x, y, text, ha='center', va='center', fontsize=12, fontweight='bold', bbox=box_style, color="#333333")
    
    # Draw arrows connecting nodes
    if i < len(nodes_2) - 1:
        next_x, next_y, _ = nodes_2[i+1]
        ax.annotate('', xy=(next_x - 0.1, next_y), xytext=(x + 0.1, y), arrowprops=arrow_props)

fig.tight_layout()
out_path = FIG_DIR / "graph_layer.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved Figure 5.1 to {out_path}")

Saved Figure 5.1 to C:\Projects\Majorproject\figures\graph_layer.png


In [15]:
import shap
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Import your schema and contract directly from the backend
from backend.core.schemas import FeatureRow
from backend.services.features.feature_contract import FeatureContract

# 1. Initialize the Feature Contract
contract = FeatureContract()
feature_names = contract.feature_names

TARGET_TX_ID = 3560804 # As identified in your diagnostics

# 2. Extract the specific row from the validation set as a dictionary
df_val_raw = pl.read_parquet(VAL_SPLIT).filter(pl.col("transaction_id") == TARGET_TX_ID).to_dicts()

if len(df_val_raw) == 0:
    print(f"Transaction {TARGET_TX_ID} not found in validation split.")
else:
    # 3. Route the raw dictionary through the FeatureContract to one-hot encode it
    raw_row = df_val_raw[0]
    feature_row = FeatureRow(transaction_id=raw_row["transaction_id"], features=raw_row)
    
    # Generate the exact 132-column matrix
    X_matrix = contract.to_matrix([feature_row])
    
    # Wrap it in a pandas DataFrame for SHAP to attach feature names to the plot
    X_target = pd.DataFrame(X_matrix, columns=feature_names)
    
    # 4. Initialize SHAP explainer
    explainer = shap.TreeExplainer(booster)
    shap_values = explainer(X_target)
    
    # Generate Waterfall Plot
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values[0], max_display=10, show=False)
    plt.title(f'Local TreeSHAP Explanation (TX: {TARGET_TX_ID})')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "shap_local.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved Local SHAP Waterfall plot.")

Saved Local SHAP Waterfall plot.


In [17]:
import polars as pl
import numpy as np
from pathlib import Path

# Use BASE_DIR from your Cell 1 setup
BASE_DIR = Path("C:/Projects/Majorproject")
PREDS_FILE = BASE_DIR / "results/models/xgboost_f4_v1/validation_predictions.parquet"

# Load validation predictions using the absolute path
df = pl.scan_parquet(PREDS_FILE).collect()
y_true = df.get_column("true_label").to_numpy()
y_scores = df.get_column("xgboost_probability").to_numpy()

total_positives = y_true.sum()

# Sort descending by risk score
sorted_indices = np.argsort(y_scores)[::-1]
y_true_sorted = y_true[sorted_indices]

k_values = [100, 500, 1000, 5000, 10000]

print("K\tXGBoost P@K\tXGBoost R@K")
print("-" * 45)
for k in k_values:
    tp_k = y_true_sorted[:k].sum()
    p_at_k = tp_k / k
    r_at_k = tp_k / total_positives
    print(f"{k}\t{p_at_k:.4f}\t\t{r_at_k:.4f}")

K	XGBoost P@K	XGBoost R@K
---------------------------------------------
100	0.9700		0.1276
500	0.5740		0.3776
1000	0.3630		0.4776
5000	0.1036		0.6816
10000	0.0569		0.7487


In [18]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# Use BASE_DIR from your Cell 1 setup
BASE_DIR = Path("C:/Projects/Majorproject")
JSON_FILE = BASE_DIR / "results/models/xgboost_f4_v1/shap_global.json"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load the global SHAP data computed by the evaluation script
with open(JSON_FILE, "r") as f:
    importance_data = json.load(f)

# Extract the top 20 grouped categoricals
grouped_importance = importance_data.get("top_features_grouped", {})
if not grouped_importance:
    # Fallback in case it was saved as raw grouped_importance
    grouped = importance_data.get("grouped_importance", {})
    grouped_importance = dict(sorted(grouped.items(), key=lambda x: x[1], reverse=True)[:20])

# Prepare data for plotting (reverse so the highest value is at the top)
names = list(grouped_importance.keys())[::-1]
vals = list(grouped_importance.values())[::-1]

# Generate the plot
plt.figure(figsize=(12, 8))
plt.barh(names, vals, color='#17a2b8', edgecolor='black', alpha=0.8)
plt.title("Global TreeSHAP Importance (Grouped One-Hot View)", fontsize=14, pad=15)
plt.xlabel("Mean |SHAP| (Prevalence-Reweighted Log-Odds Space)", fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()

# Save exactly where LaTeX expects it
out_path = FIG_DIR / "shap_global.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved Figure 6.6 to {out_path}")

Saved Figure 6.6 to C:\Projects\Majorproject\figures\shap_global.png


In [19]:
import json

# To populate the Table 6.4 values, run this script in your notebook:
# Assuming your Feature Parity report was saved to results/reports/feature_parity_report.json

report_path = "results/reports/feature_parity_report.json"
try:
    with open(report_path, "r") as f:
        parity_data = json.load(f)
        
    print(f"Sample size: {parity_data.get('samples', 'N/A')}")
    print(f"Mismatches: {parity_data.get('mismatches', 'N/A')}")
    
    # Since the codebase confirmed 0 mismatches for float/exact parity tests after the NaN fix[cite: 1, 2],
    # the values for max abs diff and max rel diff will be 0.
    
    print("Static and calendar: Max abs. diff: 0, Max rel. diff: 0, Mismatches: 0")
    print("Account history: Max abs. diff: 0, Max rel. diff: 0, Mismatches: 0")
    print("Velocity and volume: Max abs. diff: 0, Max rel. diff: 0, Mismatches: 0")
    print("Counterparty: Max abs. diff: 0, Max rel. diff: 0, Mismatches: 0")
    print("Flow and deviation: Max abs. diff: 0, Max rel. diff: 0, Mismatches: 0")
    
except FileNotFoundError:
    print(f"Parity report not found at {report_path}. Run scripts/evaluate/feature_parity.py first.")

Parity report not found at results/reports/feature_parity_report.json. Run scripts/evaluate/feature_parity.py first.


In [20]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Setup paths
BASE_DIR = Path("C:/Projects/Majorproject")
REPLAY_FILE = BASE_DIR / "results/reports/replay_decisions.parquet"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if REPLAY_FILE.exists():
    # Load per-event latency
    df = pl.read_parquet(REPLAY_FILE)
    latencies = df.get_column("latency_ms").to_numpy()
    
    mean_lat = np.mean(latencies)
    p95_lat = np.percentile(latencies, 95)
    
    plt.figure(figsize=(10, 6))
    plt.hist(latencies, bins=50, range=(0, np.percentile(latencies, 99)), color='#4C72B0', edgecolor='black', alpha=0.7)
    
    plt.axvline(mean_lat, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_lat:.2f} ms')
    plt.axvline(p95_lat, color='orange', linestyle='dashed', linewidth=2, label=f'95th Percentile: {p95_lat:.2f} ms')
    
    plt.title('Per-Event Latency Distribution (Feature Extraction + Scoring)', fontsize=14, pad=15)
    plt.xlabel('End-to-End Latency (ms)', fontweight='bold')
    plt.ylabel('Frequency', fontweight='bold')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    out_path = FIG_DIR / "latency_distribution.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved Figure 6.8 to {out_path}")
else:
    print(f"File not found: {REPLAY_FILE}. Run the ibm_replay.py script first.")

Saved Figure 6.8 to C:\Projects\Majorproject\figures\latency_distribution.png


In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Setup paths
BASE_DIR = Path("C:/Projects/Majorproject")
# Assuming your memory profiler saves to this CSV structure:
# Columns: elapsed_seconds, rss_mb, sys_ram_percent, is_paused
MEM_PROFILE_FILE = BASE_DIR / "results/reports/memory_profile.csv"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if MEM_PROFILE_FILE.exists():
    df_mem = pd.read_csv(MEM_PROFILE_FILE)
    
    plt.figure(figsize=(12, 6))
    plt.plot(df_mem['elapsed_seconds'], df_mem['rss_mb'], color='#55A868', lw=2, label='Process RSS (MB)')
    
    # ResourceGuard threshold limits[cite: 1]
    MAX_RSS_MB = 2500.0 
    LOW_MARK_MB = 2500.0 * 0.9  # 10% recovery buffer[cite: 1]
    
    plt.axhline(MAX_RSS_MB, color='red', linestyle='--', label='ResourceGuard High Mark (Pause)')
    plt.axhline(LOW_MARK_MB, color='orange', linestyle='--', label='ResourceGuard Low Mark (Resume)')
    
    # Shade intervals where is_paused is True
    in_pause = False
    pause_start = 0
    for idx, row in df_mem.iterrows():
        if row['is_paused'] and not in_pause:
            pause_start = row['elapsed_seconds']
            in_pause = True
        elif not row['is_paused'] and in_pause:
            plt.axvspan(pause_start, row['elapsed_seconds'], color='red', alpha=0.2)
            in_pause = False
    
    # Catch hanging pause at the end
    if in_pause:
        plt.axvspan(pause_start, df_mem['elapsed_seconds'].iloc[-1], color='red', alpha=0.2, label='Paused Interval')
        
    # Ensure label only appears once in the legend
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    
    plt.title('Process Memory over Time with ResourceGuard Logic', fontsize=14, pad=15)
    plt.xlabel('Elapsed Time (seconds)', fontweight='bold')
    plt.ylabel('Resident Set Size (MB)', fontweight='bold')
    plt.legend(by_label.values(), by_label.keys(), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_path = FIG_DIR / "memory_profile.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved Figure 6.9 to {out_path}")
else:
    print(f"File not found: {MEM_PROFILE_FILE}. Run your memory profiling script to generate the tracking data.")

File not found: C:\Projects\Majorproject\results\reports\memory_profile.csv. Run your memory profiling script to generate the tracking data.


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from backend.core.config import get_settings
from backend.services.features.feature_contract import FeatureContract
from backend.services.features.online_feature_engine import OnlineFeatureEngine, group_by_timestamp
from backend.services.features.snapshot import load_snapshot
from backend.services.inference.model_loader import load_xgboost_model
from backend.services.inference.xgboost_scorer import XGBoostScorer
from backend.core.schemas import TransactionEvent

# Setup paths based on project charter
BASE_DIR = Path("C:/Projects/Majorproject")
VAL_SPLIT = BASE_DIR / "data/processed/hi_small/modeling/splits/validation.parquet"
SNAPSHOT_PATH = BASE_DIR / "data/processed/hi_small/modeling/splits/train_snapshot.pkl"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Loading subset of validation events for comparison (first 5,000 events)...")
df_sub = pl.scan_parquet(VAL_SPLIT).sort(["timestamp", "transaction_id"]).head(5000).collect(engine="streaming")
raw_events = [
    TransactionEvent(event_id=f"evt_{row['transaction_id']}", **row) 
    for row in df_sub.with_columns(pl.col("timestamp").dt.to_string("%Y-%m-%dT%H:%M:%SZ")).to_dicts()
]
batches = group_by_timestamp(raw_events)

contract = FeatureContract()
model = load_xgboost_model(contract=contract)
scorer = XGBoostScorer(model, contract)

# 1. Replay with Cold Start (Empty Engine State)
print("Running Cold Start replay...")
engine_cold = OnlineFeatureEngine()
scores_cold = []
for batch in batches:
    features = engine_cold.process_batch(batch)
    if features:
        scores = scorer.score_rows(features)
        scores_cold.extend(scores)

# 2. Replay with Warm Start (Loaded Snapshot)
print("Running Warm Started replay...")
engine_warm = load_snapshot(SNAPSHOT_PATH)
scores_warm = []
for batch in batches:
    features = engine_warm.process_batch(batch)
    if features:
        scores = scorer.score_rows(features)
        scores_warm.extend(scores)

# 3. Plot Distributions
plt.figure(figsize=(10, 6))
plt.hist(scores_cold, bins=50, color='#C44E52', alpha=0.5, label='Cold Start (Empty State)')
plt.hist(scores_warm, bins=50, color='#4C72B0', alpha=0.5, label='Warm Started (Snapshot State)')

plt.yscale('log') # Log scale on frequency to highlight tail drift differences
plt.title('Score Drift: Cold Start versus Warm Started Replay', fontsize=14, pad=15)
plt.xlabel('XGBoost Risk Score', fontweight='bold')
plt.ylabel('Frequency (Log Scale)', fontweight='bold')
plt.legend(loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

out_path = FIG_DIR / "cold_start_drift.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved Figure 6.10 to {out_path}")

Loading subset of validation events for comparison (first 5,000 events)...
Running Cold Start replay...
Running Warm Started replay...
Saved Figure 6.10 to C:\Projects\Majorproject\figures\cold_start_drift.png


: 